# ActivIQ: Qwen3.5-9B zero-shot benchmark
Runtime > Change runtime type > **GPU** (A100/L4 best; T4 works only with 4-bit).
Runs `scripts/evaluate_lora.py` on the 120-utterance held-out test split and prints intent accuracy / macro-F1 / per-class P-R-F1 / confusion matrix.
Test entity labels are human reviewed, so entity F1 is reported too.

In [ ]:
# 1. Clone the repo (private repo? set GITHUB_TOKEN via Colab Secrets and uncomment the token line)
import os
REPO = "jiakai2002/ActivIQ"
BRANCH = "lora-finetuning"
# from google.colab import userdata; os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
tok = os.environ.get("GITHUB_TOKEN")
url = f"https://{tok+'@' if tok else ''}github.com/{REPO}.git"
!rm -rf ActivIQ && git clone -q --depth 1 -b {BRANCH} {url} ActivIQ
%cd ActivIQ
!ls scripts/evaluate_lora.py src/lora.py

In [ ]:
# 2. Dependencies (keep Colab's own torch)
!pip install -q "transformers>=5.3,<6" "peft>=0.13,<1" "accelerate>=1,<2" safetensors bitsandbytes pandas spacy
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# 3. Download Qwen3.5-9B (~19 GB, a few minutes). Pin the revision for reproducibility.
from huggingface_hub import snapshot_download
path = snapshot_download("Qwen/Qwen3.5-9B", local_dir="models/qwen3.5-9b")
print(path)

In [ ]:
# 4. Benchmark. 4bit = fits any GPU >=12 GB; change to "none" on A100 for bf16.
QUANT = "4bit"
!python scripts/evaluate_lora.py --model-path models/qwen3.5-9b --device cuda --quantization {QUANT} \
    --skip-baseline --output-dir reports/evaluation/qwen9b_zeroshot

In [ ]:
# 5. Full metrics
import json
r = json.load(open("reports/evaluation/qwen9b_zeroshot/lora_comparison.json"))["lora"]
i = r["intent"]
print(f"Intent accuracy {i['accuracy']:.4f} | macro-F1 {i['macro_f1']:.4f} | weighted-F1 {i['weighted_f1']:.4f}")
import pandas as pd
display(pd.DataFrame(i["per_class"]).T[["precision","recall","f1","support"]])
display(pd.DataFrame(i["confusion_gold_by_predicted"]).T)
print("Entities:", {k: r["entities"][k] for k in r["entities"] if k in ("status","scored","total")} if "micro" not in r["entities"] else r["entities"]["micro"])
print("Invalid outputs:", r["invalid_outputs"], "| latency:", r["latency_seconds"])

In [ ]:
# 6. Download results
!zip -qr qwen9b_results.zip reports/evaluation/qwen9b_zeroshot
from google.colab import files; files.download("qwen9b_results.zip")